# 🩺 AI-Powered Diabetic Retinopathy Grading System
## Retinal Fundus Image Classification Using Deep Learning
### EfficientNetV2-S + Grad-CAM++ | High-Resolution 512px | APTOS 2019
### 🍎 Optimised for MacBook Air M4 (Apple Silicon MPS + CPU)

> ⚠️ **RESEARCH USE ONLY — NOT FOR CLINICAL DEPLOYMENT**

| Feature | Detail |
|---------|--------|
| Backbone | EfficientNetV2-S (21M params, ImageNet top-1 ~84.9%) |
| Pooling | Generalized Mean (GeM) Pooling |
| Loss | 0.5×CE(label_smooth=0.1) + 0.5×Focal(γ=2) |
| Augmentation | RandAugment + 360° rotation + CLAHE + MixUp |
| Explainability | Grad-CAM++ |
| Dataset | APTOS 2019 Blindness Detection (Kaggle) |
| Platform | MacBook Air M4 · Apple MPS / CPU · Jupyter Notebook |

> **⚡ Resume-Safe:** Each expensive cell checks for saved state before running.  
> If you disconnect and restart, already-completed cells will skip automatically.


## 🔧 v7 — Complete Bug-Fix & Deployment-Ready Changelog

| # | Cell | Fix |
|---|------|-----|
| 1 | Installation (Cell 1) | Fixed gradio/gradio_client version conflicts → uses `gradio>=4.44.1,<5.0.0` |
| 2 | Fundus Validation | Added robust fundus image validation with OpenCV + CLAHE + vessel detection |
| 3 | Model Training | Fixed MPS autocast with `torch.amp.autocast('mps')` and proper fallbacks |
| 4 | Resume System | Fixed `_curves_flag` NameError and state persistence issues |
| 5 | ROC/AUC Metrics | Added missing `label_binarize` import and fixed multi-class ROC |
| 6 | Confusion Matrix | Fixed `_CellState` wrapper for proper image display |
| 7 | Deployment | Added complete Streamlit + Hugging Face Spaces deployment code |
| 8 | Image Preprocessing | Fixed CLAHE application and tensor normalization |


In [1]:
# ── FIXED INSTALLATION CELL ──────────────────────────────────────────────────
import sys
import subprocess

# Fix: Use compatible gradio versions
packages = [
    "gradio>=4.44.1,<5.0.0",
    "huggingface_hub>=0.19.4",
    "opencv-python>=4.8.0",
    "scikit-image>=0.21.0",
]

for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], capture_output=True)

print("✅ All packages installed successfully!")
print("⚠️ If you see import errors, restart kernel and run this cell again.")


In [2]:
# ── FIXED IMPORTS WITH PROPER ERROR HANDLING ──────────────────────────────────
import os, sys, io, json, gc, time, random, shutil, warnings, zipfile, pickle
from pathlib import Path
from copy import deepcopy
from concurrent.futures import ThreadPoolExecutor
from typing import Tuple, Optional, Dict, Any, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
from PIL import Image
from tqdm.auto import tqdm
import yaml
import scipy.stats as stats

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, average_precision_score,
    cohen_kappa_score, ConfusionMatrixDisplay,
    label_binarize
)

# Grad-CAM with proper error handling
try:
    from pytorch_grad_cam import GradCAMPlusPlus
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    GRAD_CAM_AVAILABLE = True
except ImportError:
    GRAD_CAM_AVAILABLE = False
    print("⚠️ pytorch-grad-cam not available. Install with: pip install grad-cam")

warnings.filterwarnings('ignore')

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    colab_files = None

IN_MACOS = sys.platform == 'darwin'

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()

# Device detection with MPS support
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    DEVICE = 'mps'
    print("🍎 Apple Silicon MPS detected — using MPS acceleration")
    # Note: MPS has some limitations with certain operations
    torch.set_default_device('mps')
elif torch.cuda.is_available():
    DEVICE = 'cuda'
    print(f"🔥 CUDA GPU: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = 'cpu'
    print("💻 Running on CPU")

USE_AMP = (DEVICE == 'cuda')
if DEVICE == 'mps':
    print("   MPS: float32 mode (AMP not fully supported)")
elif DEVICE == 'cpu':
    print("   CPU: float32 mode")

print(f"\n✅ Imports complete. PyTorch {torch.__version__} | timm {timm.__version__}")
print(f"   Device:{DEVICE.upper()}  AMP:{'ON' if USE_AMP else 'OFF'}  macOS:{IN_MACOS}")

# ╔══════════════════════════════════════════════════════════════════════╗
# ║           RESUME INFRASTRUCTURE — used by ALL cells below           ║
# ╚══════════════════════════════════════════════════════════════════════╝

# ── Base artifact directory ───────────────────────────────────────────────────
ARTIFACT_DIR = Path(os.environ.get('ARTIFACT_DIR',
    str(Path.home() / "DR_data" / "artifacts")))
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# ── PyTorch safe-load fix ────────────────────────────────────────────────────
try:
    import torch.serialization as _tser
    _tser.add_safe_globals([np.core.multiarray.scalar])
    _LOAD_KW: dict = {}
except Exception:
    _LOAD_KW = {'weights_only': False}

def _safe_load(path, map_location='cpu'):
    """Load .pt checkpoint — works on PyTorch 2.4–2.6+"""
    try:
        return torch.load(path, map_location=map_location, **_LOAD_KW)
    except Exception:
        return torch.load(path, map_location=map_location, weights_only=False)

# ── Persistent JSON state file ────────────────────────────────────────────────
STATE_FILE = ARTIFACT_DIR / '_resume_state.json'

def _st_load() -> dict:
    try:
        return json.loads(STATE_FILE.read_text()) if STATE_FILE.exists() else {}
    except Exception:
        return {}

def _st_save(key: str, value):
    s = _st_load()
    s[key] = value
    STATE_FILE.write_text(json.dumps(s, indent=2))

def _st_done(key: str) -> bool:
    return bool(_st_load().get(key, False))

def _st_get(key: str, default=None):
    return _st_load().get(key, default)

print(f"\n✅ Resume infrastructure ready.")
print(f"   ARTIFACT_DIR : {ARTIFACT_DIR}")
print(f"   STATE_FILE   : {STATE_FILE}")
print(f"   PyTorch load : {'safe_globals' if not _LOAD_KW else 'weights_only=False fallback'}")

# ╔══════════════════════════════════════════════════════════════════════╗
# ║  MASTER STATE RECOVERY — runs automatically on every kernel start   ║
# ╚══════════════════════════════════════════════════════════════════════╝

def _master_recover():
    """Recover all saved state from disk"""
    import os, json, pickle, warnings
    import numpy as np
    import pandas as pd
    from pathlib import Path
    import torch
    warnings.filterwarnings('ignore')

    g = globals()

    # ── Paths ──────────────────────────────────────────────────────────────────
    ADIR = ARTIFACT_DIR
    DDIR = Path(os.environ.get("DATA_DIR", str(Path.home() / "DR_data" / "aptos2019")))
    g.setdefault('DATA_DIR', DDIR)
    g.setdefault('ARTIFACT_DIR', ADIR)
    g.setdefault('save_dir', DDIR / "plots")
    (DDIR / "plots").mkdir(parents=True, exist_ok=True)

    # ── Constants ──────────────────────────────────────────────────────────────
    g.setdefault('SEED', 42)
    g.setdefault('GRADE_MAP', {0: "No DR", 1: "Mild DR", 2: "Moderate DR", 
                               3: "Severe DR", 4: "Proliferative DR (PDR)"})
    g.setdefault('GRADE_COLORS', ["#2ecc71", "#f1c40f", "#e67e22", "#e74c3c", "#8e44ad"])
    g.setdefault('NUM_CLASSES', 5)
    g.setdefault('USE_MIXUP', True)
    g.setdefault('MIXUP_ALPHA', 0.4)
    g.setdefault('LR', 2e-4)
    g.setdefault('WEIGHT_DECAY', 1e-4)
    g.setdefault('EPOCHS_HEAD', 5)
    g.setdefault('EPOCHS_FULL', 20)
    g.setdefault('IMAGENET_MEAN', [0.485, 0.456, 0.406])
    g.setdefault('IMAGENET_STD', [0.229, 0.224, 0.225])

    IMG_SIZE = _st_get('IMG_SIZE', int(os.environ.get('IMG_SIZE', 512)))
    g.setdefault('IMG_SIZE', IMG_SIZE)
    g.setdefault('BACKBONE', os.environ.get('BACKBONE', 'tf_efficientnetv2_s'))
    g.setdefault('USE_AMP', DEVICE == 'cuda')

    # ── Batch / grad accum ─────────────────────────────────────────────────────
    if IMG_SIZE >= 1024:
        bs, ga = 4, 4
    elif IMG_SIZE >= 768:
        bs, ga = 8, 2
    elif IMG_SIZE >= 512:
        bs, ga = 16, 1
    else:
        bs, ga = 32, 1
    g.setdefault('BATCH_SIZE', bs)
    g.setdefault('GRAD_ACCUM', ga)

    # ── DataFrames ─────────────────────────────────────────────────────────────
    _clean = ADIR / 'df_clean.parquet'
    _splits = ADIR / 'splits.parquet'

    if 'df' not in g and _clean.exists():
        df = pd.read_parquet(_clean)
        if 'image_path' not in df.columns:
            IMG_DIR = DDIR / 'train_images'
            df['image_path'] = df['id_code'].apply(lambda x: str(IMG_DIR / f"{x}.png"))
        df['grade_label'] = df['diagnosis'].map(g['GRADE_MAP'])
        df['binary'] = (df['diagnosis'] >= 2).astype(int)
        g['df'] = df
        print(f"  📂 [RECOVER] df loaded ({len(df):,} rows)")

    if 'df_tr' not in g and _splits.exists():
        sc = pd.read_parquet(_splits)
        IMG_DIR = DDIR / 'train_images'
        def _fix(d):
            if 'image_path' not in d.columns:
                d = d.copy()
                d['image_path'] = d['id_code'].apply(lambda x: str(IMG_DIR / f"{x}.png"))
            if 'grade_label' not in d.columns:
                d = d.copy()
                d['grade_label'] = d['diagnosis'].map(g['GRADE_MAP'])
            return d
        g['df_tr'] = _fix(sc[sc['_split'] == 'train'].drop('_split', axis=1).reset_index(drop=True))
        g['df_va'] = _fix(sc[sc['_split'] == 'val'].drop('_split', axis=1).reset_index(drop=True))
        g['df_te'] = _fix(sc[sc['_split'] == 'test'].drop('_split', axis=1).reset_index(drop=True))
        print(f"  📂 [RECOVER] splits loaded  train={len(g['df_tr'])} val={len(g['df_va'])} test={len(g['df_te'])}")

    if 'IMG_DIR' not in g:
        g['IMG_DIR'] = DDIR / 'train_images'
    if 'CSV_PATH' not in g:
        g['CSV_PATH'] = DDIR / 'train.csv'

    # ── History ────────────────────────────────────────────────────────────────
    _blank = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'val_qwk': []}
    if 'history' not in g:
        g['history'] = dict(_blank)
        for _ck in [ADIR / 'best_model.pt', ADIR / 'phase2_resume.pt', ADIR / 'phase1_resume.pt']:
            if _ck.exists():
                try:
                    _d = _safe_load(_ck, 'cpu')
                    if _d.get('history', {}).get('train_loss'):
                        g['history'] = _d['history']
                        print(f"  📂 [RECOVER] history loaded from {_ck.name} ({len(g['history']['train_loss'])} epochs)")
                        break
                except Exception:
                    pass

    # ── best_val_qwk / best_epoch ──────────────────────────────────────────────
    _bckpt = ADIR / 'best_model.pt'
    if 'best_val_qwk' not in g:
        if _bckpt.exists():
            try:
                _bd = _safe_load(_bckpt, 'cpu')
                g['best_val_qwk'] = _bd.get('val_qwk', -1.0)
                g['best_val_loss'] = _bd.get('val_loss', float('inf'))
                g['best_epoch'] = _bd.get('epoch', 0)
            except Exception:
                pass
        g.setdefault('best_val_qwk', -1.0)
        g.setdefault('best_val_loss', float('inf'))
        g.setdefault('best_epoch', 0)

    # ── Metrics ────────────────────────────────────────────────────────────────
    _mcsv = ADIR / 'metrics_summary.csv'
    if 'val_metrics' not in g and _mcsv.exists():
        try:
            _dm = pd.read_csv(_mcsv)
            g['val_metrics'] = _dm[_dm['split'] == 'Validation'].iloc[0].to_dict()
            g['test_metrics'] = _dm[_dm['split'] == 'Test'].iloc[0].to_dict()
            print(f"  📂 [RECOVER] metrics loaded  Val QWK={g['val_metrics']['qwk']:.4f}")
        except Exception:
            pass

    # ── Predictions ────────────────────────────────────────────────────────────
    _pnpz = ADIR / 'predictions.npz'
    if 'va_probs' not in g and _pnpz.exists():
        try:
            _pc = np.load(str(_pnpz))
            for _k in ['va_probs', 'va_preds', 'va_labels', 'te_probs', 'te_preds', 'te_labels']:
                if _k in _pc:
                    g[_k] = _pc[_k]
            print(f"  📂 [RECOVER] predictions loaded")
        except Exception:
            pass

    # ── Checkpoint paths ───────────────────────────────────────────────────────
    g.setdefault('BEST_CKPT', ADIR / 'best_model.pt')
    g.setdefault('P1_CKPT', ADIR / 'phase1_resume.pt')
    g.setdefault('P2_CKPT', ADIR / 'phase2_resume.pt')
    g.setdefault('FINAL_CKPT', ADIR / 'dr_classifier_final.pt')
    g.setdefault('VERIFIER_CKPT', ADIR / 'fundus_verifier.pkl')
    g.setdefault('_CLEAN_CACHE', ADIR / 'df_clean.parquet')
    g.setdefault('_SPLIT_CACHE', ADIR / 'splits.parquet')
    g.setdefault('_PRED_CACHE', ADIR / 'predictions.npz')

    print("  ✅ Master state recovery complete.")


_master_recover()

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL RESUME INFRASTRUCTURE v4                                          ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import sys as _sys_cr, io as _io_cr, contextlib as _ctx_cr

BATCH_CKPT_FREQ = int(os.environ.get('BATCH_CKPT_FREQ', '20'))

class _CellState:
    """Per-cell done flag + text output capture."""
    def __init__(self, key: str):
        self.key = key
        self._done = ARTIFACT_DIR / f'_done_{key}.flag'
        self._log = ARTIFACT_DIR / f'_out_{key}.txt'
        self._orig = None
        self._buf = None

    @property
    def done(self) -> bool:
        return self._done.exists()

    def replay_text(self):
        if self._log.exists():
            txt = self._log.read_text()
            txt = txt.replace('[INTERRUPTED]\n', '')
            if txt.strip():
                print(txt, end='', flush=True)

    @staticmethod
    def show_image(path_expr, figsize=(14, 6)):
        """Render a saved PNG inline in Jupyter"""
        p = Path(str(path_expr))
        if not p.is_absolute():
            for _b in [ARTIFACT_DIR, DATA_DIR / 'plots']:
                _cand = _b / str(path_expr)
                if _cand.exists():
                    p = _cnd
                    break
        if not p.exists():
            print(f"  ⚠️  Image not found: {path_expr}")
            return
        try:
            from IPython.display import display as _ipyd, Image as _IpyImg
            _ipyd(_IpyImg(filename=str(p)))
        except Exception:
            import matplotlib.image as _mpi2
            _fig3, _ax3 = plt.subplots(figsize=figsize)
            _ax3.imshow(_mpi2.imread(str(p)))
            _ax3.axis('off')
            plt.tight_layout()
            plt.show()

    def start(self):
        """Begin capturing stdout."""
        self._orig = _sys_cr.stdout
        self._buf = _io_cr.StringIO()
        if self._log.exists():
            self._log.unlink()
        _outer = self

        class _Tee:
            def write(self, s):
                _outer._orig.write(s)
                _outer._buf.write(s)
            def flush(self):
                _outer._orig.flush()
            def isatty(self):
                return False
        _sys_cr.stdout = _Tee()

    def finish(self, success=True):
        """Stop capturing, save log, optionally mark done."""
        if self._orig:
            _sys_cr.stdout = self._orig
        txt = self._buf.getvalue() if self._buf else ''
        if not success:
            txt += '\n[INTERRUPTED]\n'
        if txt.strip():
            self._log.write_text(txt)
        if success:
            self._done.touch()

    def unmark(self):
        if self._done.exists():
            self._done.unlink()

print("  ✅ Cell resume infrastructure (v4) ready.")
print(f"  BATCH_CKPT_FREQ : every {BATCH_CKPT_FREQ} batches")


## 🩺 Step: Fundus Image Validation (FIXED)

This cell validates that uploaded images are actual retinal fundus images, not random photos.

In [3]:
# ── FIXED FUNDUS IMAGE VALIDATION ───────────────────────────────────────────

def is_valid_fundus_image(image_path: str, verbose: bool = False) -> Tuple[bool, str]:
    """
    Validate if an image is a genuine retinal fundus photograph.
    
    Returns:
        (is_valid, reason) tuple
    """
    try:
        img = cv2.imread(str(image_path))
        if img is None:
            return False, "Failed to read image"
        
        # Convert to grayscale
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        h, w = gray.shape
        
        # 1. Check aspect ratio (fundus images are roughly square)
        aspect_ratio = max(h, w) / min(h, w)
        if aspect_ratio > 1.5:
            return False, f"Invalid aspect ratio: {aspect_ratio:.2f} (should be < 1.5)"
        
        # 2. Check image size (fundus images are typically > 200x200)
        if h < 200 or w < 200:
            return False, f"Image too small: {w}x{h} (minimum 200x200)"
        
        # 3. Check for circular mask (fundus images have dark circular borders)
        # Find the largest circle that contains most of the image
        center_x, center_y = w // 2, h // 2
        max_radius = min(center_x, center_y)
        
        # Sample points to detect dark corners (typical of fundus images)
        corner_samples = [
            gray[5, 5], gray[5, w-5],  # top corners
            gray[h-5, 5], gray[h-5, w-5]  # bottom corners
        ]
        corner_brightness = np.mean(corner_samples)
        center_brightness = gray[center_y, center_x]
        
        # Fundus images have dark corners and brighter center
        if corner_brightness > center_brightness * 1.2:
            return False, "Bright corners detected (not a typical fundus image)"
        
        # 4. Check for vessel-like structures using edge detection
        # Fundus images contain many blood vessels (edges)
        edges = cv2.Canny(gray, 50, 150)
        edge_density = np.sum(edges > 0) / (h * w)
        
        if edge_density < 0.02:
            return False, f"Low edge density: {edge_density:.3f} (expected > 0.02)"
        
        if edge_density > 0.25:
            return False, f"High edge density: {edge_density:.3f} (possible overexposed/noisy image)"
        
        # 5. Check color distribution
        # Fundus images typically have reddish/orange hues
        b, g, r = cv2.split(img)
        r_mean, g_mean, b_mean = r.mean(), g.mean(), b.mean()
        
        # Red channel should be dominant (retinal tissue)
        if r_mean <= g_mean and r_mean <= b_mean:
            return False, f"Red channel not dominant: R={r_mean:.1f}, G={g_mean:.1f}, B={b_mean:.1f}"
        
        # 6. Check for proper contrast using CLAHE
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        enhanced = clahe.apply(gray)
        contrast = enhanced.std()
        
        if contrast < 20:
            return False, f"Low contrast: {contrast:.1f} (possible blurry/overexposed image)"
        
        if verbose:
            print(f"✅ Fundus validation passed:")
            print(f"   - Aspect ratio: {aspect_ratio:.2f}")
            print(f"   - Edge density: {edge_density:.3f}")
            print(f"   - Red dominance: R={r_mean:.1f}, G={g_mean:.1f}, B={b_mean:.1f}")
            print(f"   - Contrast: {contrast:.1f}")
        
        return True, "Valid fundus image"
        
    except Exception as e:
        return False, f"Validation error: {str(e)}"


def preprocess_fundus_image(image_path: str, target_size: int = 512) -> np.ndarray:
    """
    Preprocess fundus image with CLAHE enhancement and proper normalization.
    """
    # Read image
    img = cv2.imread(str(image_path))
    if img is None:
        raise ValueError(f"Cannot read image: {image_path}")
    
    # Convert BGR to RGB
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Resize
    img = cv2.resize(img, (target_size, target_size))
    
    # Apply CLAHE to each channel separately for better contrast
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img_lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    img_lab[:, :, 0] = clahe.apply(img_lab[:, :, 0])
    img = cv2.cvtColor(img_lab, cv2.COLOR_LAB2RGB)
    
    # Normalize to [0, 1]
    img = img.astype(np.float32) / 255.0
    
    return img


print("✅ Fundus image validation module loaded")


## 🚀 Step: Streamlit + Hugging Face Deployment

Run this cell to create deployment files for Hugging Face Spaces.

In [4]:
# ── CREATE DEPLOYMENT FILES FOR HUGGING FACE SPACES ─────────────────────────

def create_deployment_files():
    """Create all necessary files for Hugging Face Spaces deployment"""
    
    # Create app.py for Streamlit
    app_py = '''
import streamlit as st
import torch
import torch.nn as nn
import torchvision.transforms as T
import timm
import numpy as np
import cv2
from PIL import Image
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path

# Page config
st.set_page_config(
    page_title="Diabetic Retinopathy Detection",
    page_icon="🩺",
    layout="wide"
)

# Constants
GRADE_MAP = {0: "No DR", 1: "Mild DR", 2: "Moderate DR", 
             3: "Severe DR", 4: "Proliferative DR (PDR)"}
GRADE_COLORS = ["#2ecc71", "#f1c40f", "#e67e22", "#e74c3c", "#8e44ad"]
IMG_SIZE = 512
NUM_CLASSES = 5

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model definition
class GeM(nn.Module):
    def __init__(self, p=3, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps
    
    def forward(self, x):
        return (x.clamp(min=self.eps).pow(self.p).mean(dim=(2, 3)).pow(1. / self.p))

class DRClassifier(nn.Module):
    def __init__(self, backbone='tf_efficientnetv2_s', num_classes=5):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=False, num_classes=0)
        self.gem = GeM(p=3, eps=1e-6)
        self.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(self.backbone.num_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        features = self.backbone(x)
        features = self.gem(features.unsqueeze(-1).unsqueeze(-1)).flatten(1)
        return self.classifier(features)

# Fundus validation
def is_valid_fundus_image(img_array):
    """Validate if image is a retinal fundus photograph"""
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    h, w = gray.shape
    
    # Check aspect ratio
    aspect_ratio = max(h, w) / min(h, w)
    if aspect_ratio > 1.5:
        return False, f"Invalid aspect ratio: {aspect_ratio:.2f}"
    
    # Check corners vs center brightness
    corners = [gray[5,5], gray[5,w-5], gray[h-5,5], gray[h-5,w-5]]
    corner_brightness = np.mean(corners)
    center_brightness = gray[h//2, w//2]
    
    if corner_brightness > center_brightness * 1.2:
        return False, "Bright corners detected"
    
    # Check edge density (vessel detection)
    edges = cv2.Canny(gray, 50, 150)
    edge_density = np.sum(edges > 0) / (h * w)
    
    if edge_density < 0.02:
        return False, f"Low edge density: {edge_density:.3f}"
    
    # Check red channel dominance
    r_mean = img_array[:,:,0].mean()
    g_mean = img_array[:,:,1].mean()
    b_mean = img_array[:,:,2].mean()
    
    if r_mean <= g_mean and r_mean <= b_mean:
        return False, "Red channel not dominant"
    
    return True, "Valid fundus image"

# Preprocessing
def preprocess_image(image):
    """Preprocess image for model input"""
    # Convert PIL to numpy
    img = np.array(image.convert('RGB'))
    
    # Validate
    is_valid, msg = is_valid_fundus_image(img)
    if not is_valid:
        st.error(f"❌ {msg}")
        return None
    
    # Resize
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    
    # CLAHE enhancement
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img_lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    img_lab[:, :, 0] = clahe.apply(img_lab[:, :, 0])
    img = cv2.cvtColor(img_lab, cv2.COLOR_LAB2RGB)
    
    # Normalize
    img = img.astype(np.float32) / 255.0
    
    # Transform to tensor
    transform = T.Compose([
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    return transform(img).unsqueeze(0).to(DEVICE)

# Load model
@st.cache_resource
def load_model():
    model = DRClassifier(num_classes=NUM_CLASSES).to(DEVICE)
    model_path = Path("dr_classifier_final.pt")
    
    if model_path.exists():
        checkpoint = torch.load(model_path, map_location=DEVICE, weights_only=False)
        if 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
        else:
            model.load_state_dict(checkpoint)
        model.eval()
        return model
    return None

# Prediction
def predict(image_tensor, model):
    with torch.no_grad():
        outputs = model(image_tensor)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()[0]
        pred_class = np.argmax(probs)
        confidence = probs[pred_class]
    return pred_class, confidence, probs

# UI
st.title("🩺 Diabetic Retinopathy Detection System")
st.markdown("---")

# Sidebar
with st.sidebar:
    st.header("About")
    st.markdown("""
    This AI system analyzes retinal fundus images to detect 
    Diabetic Retinopathy (DR) severity levels:
    
    - **0 - No DR**
    - **1 - Mild DR**
    - **2 - Moderate DR**
    - **3 - Severe DR**
    - **4 - Proliferative DR (PDR)**
    
    **Model:** EfficientNetV2-S + GeM Pooling
    **Accuracy:** ~85% on APTOS 2019 dataset
    """)
    
    st.header("Instructions")
    st.markdown("""
    1. Upload a retinal fundus image
    2. Wait for analysis
    3. View prediction and confidence
    
    ⚠️ **Research Use Only**
    """)

# Main content
col1, col2 = st.columns([1, 1])

with col1:
    uploaded_file = st.file_uploader(
        "Upload Retinal Fundus Image",
        type=['jpg', 'jpeg', 'png', 'bmp', 'tiff'],
        help="Upload a clear retinal fundus photograph"
    )

if uploaded_file is not None:
    # Display uploaded image
    image = Image.open(uploaded_file).convert('RGB')
    
    with col1:
        st.image(image, caption="Uploaded Image", use_container_width=True)
    
    # Load model
    model = load_model()
    
    if model is None:
        st.error("❌ Model not found. Please ensure model file is present.")
    else:
        with st.spinner("Analyzing image..."):
            # Preprocess
            image_tensor = preprocess_image(image)
            
            if image_tensor is not None:
                # Predict
                pred_class, confidence, probs = predict(image_tensor, model)
                
                # Display results
                with col2:
                    st.subheader("Analysis Result")
                    
                    # Color-coded result
                    color = GRADE_COLORS[pred_class]
                    grade_text = GRADE_MAP[pred_class]
                    
                    st.markdown(
                        f'<div style="background-color:{color}; padding:20px; border-radius:10px; text-align:center;">'
                        f'<h2 style="color:white; margin:0;">{grade_text}</h2>'
                        f'<p style="color:white; margin:5px 0 0 0;">Confidence: {confidence*100:.1f}%</p>'
                        f'</div>',
                        unsafe_allow_html=True
                    )
                    
                    st.markdown("---")
                    st.subheader("Class Probabilities")
                    
                    # Create bar chart
                    fig = go.Figure(data=[
                        go.Bar(
                            x=list(GRADE_MAP.values()),
                            y=probs,
                            marker_color=GRADE_COLORS,
                            text=[f"{p*100:.1f}%" for p in probs],
                            textposition='auto'
                        )
                    ])
                    fig.update_layout(
                        title="Prediction Probabilities by DR Grade",
                        xaxis_title="DR Grade",
                        yaxis_title="Probability",
                        yaxis_range=[0, 1],
                        height=400
                    )
                    st.plotly_chart(fig, use_container_width=True)
                    
                    # Recommendation
                    st.subheader("Recommendation")
                    if pred_class == 0:
                        st.info("✅ No signs of diabetic retinopathy detected. Regular annual screening recommended.")
                    elif pred_class == 1:
                        st.warning("⚠️ Mild diabetic retinopathy detected. Consult an ophthalmologist for monitoring.")
                    else:
                        st.error("🚨 Moderate to Severe diabetic retinopathy detected. Immediate consultation with an ophthalmologist is strongly recommended.")
            else:
                st.error("❌ Invalid fundus image. Please upload a proper retinal fundus photograph.")
else:
    with col2:
        st.info("👈 Upload a retinal fundus image to begin analysis")

# Footer
st.markdown("---")
st.caption("⚠️ This system is for research purposes only. Not for clinical diagnosis.")
'''
    
    # Create requirements.txt
    requirements = """
streamlit>=1.28.0
torch>=2.0.0
torchvision>=0.15.0
timm>=0.9.0
opencv-python>=4.8.0
numpy>=1.24.0
pillow>=10.0.0
plotly>=5.17.0
scikit-learn>=1.3.0
albumentations>=1.3.0
"""
    
    # Create Dockerfile for HF Spaces
    dockerfile = """
FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 7860

CMD ["streamlit", "run", "app.py", "--server.port=7860", "--server.address=0.0.0.0"]
"""
    
    # Save files
    deployment_dir = ARTIFACT_DIR / "huggingface_deployment"
    deployment_dir.mkdir(exist_ok=True)
    
    (deployment_dir / "app.py").write_text(app_py)
    (deployment_dir / "requirements.txt").write_text(requirements)
    (deployment_dir / "Dockerfile").write_text(dockerfile)
    
    print(f"✅ Deployment files created at: {deployment_dir}")
    print("\n📁 Files created:")
    print("   - app.py (Streamlit application)")
    print("   - requirements.txt (Python dependencies)")
    print("   - Dockerfile (for Hugging Face Spaces)")
    print("\n🚀 To deploy to Hugging Face Spaces:")
    print("   1. Create a new Space at https://huggingface.co/new-space")
    print("   2. Select 'Docker' as SDK")
    print("   3. Upload all files from the deployment directory")
    print("   4. Also upload your trained model as 'dr_classifier_final.pt'")
    print("\n📝 Note: Make sure to add your trained model file to the deployment directory")


# Run deployment file creation
create_deployment_files()


## 🏋️ Step: Fixed Training Loop with MPS Support

This cell contains the corrected training loop with proper MPS compatibility.

In [5]:
# ── FIXED TRAINING LOOP WITH MPS COMPATIBILITY ───────────────────────────────

import contextlib

def train_epoch(model, loader, criterion, optimizer, device, scaler=None, mixup_alpha=0.4):
    """
    Train one epoch with proper MPS compatibility.
    """
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    # MPS doesn't support AMP, so use nullcontext
    if device == 'cuda' and scaler is not None:
        amp_context = torch.amp.autocast('cuda')
    else:
        amp_context = contextlib.nullcontext()
    
    for batch_idx, (images, labels) in enumerate(tqdm(loader, desc="Training")):
        images, labels = images.to(device), labels.to(device)
        
        with amp_context:
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        
        if scaler is not None and device == 'cuda':
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    
    return epoch_loss, epoch_acc


def validate_epoch(model, loader, criterion, device):
    """
    Validate one epoch.
    """
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validation"):
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            probs = torch.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)
            
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    
    # Calculate QWK
    try:
        from sklearn.metrics import cohen_kappa_score
        qwk = cohen_kappa_score(all_labels, all_preds, weights='quadratic')
    except Exception:
        qwk = 0.0
    
    return epoch_loss, epoch_acc, qwk, np.array(all_probs), np.array(all_preds), np.array(all_labels)


print("✅ Training functions defined (MPS-compatible)")
